# Notebook 02 – Khai phá Luật kết hợp (Association Mining)

**Kỹ thuật:** FP-Growth (nhanh hơn Apriori cho dataset lớn)  
**Lý do chọn:** Dataset thời tiết có nhiều điều kiện đồng xuất hiện (nhiệt độ, độ ẩm, gió, tầm nhìn),
phù hợp rời rạc hoá rồi tìm pattern.  

**Pipeline:**
1. Rời rạc hoá các biến liên tục → bins (Cold/Cool/Warm/Hot, Dry/Moderate/Humid/VeryHumid, ...)
2. Tạo basket (mỗi bản ghi = 1 transaction)
3. FP-Growth với min_support, min_confidence, min_lift từ `params.yaml`
4. Phân tích luật theo mùa

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import matplotlib
matplotlib.use('Agg')
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from src.data.loader import load_config, load_processed_data
from src.features.builder import FeatureBuilder
from src.mining.association import AssociationMiner
from src.visualization import plots
from src.evaluation.report import save_table

cfg = load_config('../configs/params.yaml')
plots.FIG_DIR = '../outputs/figures'
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

In [2]:
df = load_processed_data('../data/processed/weather_processed.parquet')
print(f'Loaded: {df.shape}')

[loader] Loaded processed data: (95150, 16)
Loaded: (95150, 16)


## 2.1 Rời rạc hoá & Tạo basket

In [3]:
builder = FeatureBuilder(cfg)
basket = builder.build_basket_for_association(df)
print(f'Basket shape: {basket.shape}')
print(f'Số items (cột): {basket.shape[1]}')
basket.head(3)

Basket shape: (95150, 25)
Số items (cột): 25


,Temp_bin=Cold,Temp_bin=Cool,Temp_bin=Hot,Temp_bin=Warm,Humid_bin=Dry,Humid_bin=Humid,Humid_bin=Moderate,Humid_bin=VeryHumid,Humid_bin=nan,Wind_bin=Breezy,...,Vis_bin=ModVis,Press_bin=HighPress,Press_bin=LowPress,Press_bin=NormPress,Press_bin=VHighPress,WeatherType=Clear,WeatherType=Cloudy,WeatherType=Foggy,WeatherType=Rainy,WeatherType=Windy
0,False,True,False,False,False,False,False,True,False,True,...,False,True,False,False,False,False,True,False,False,False
1,False,True,False,False,False,False,False,True,False,True,...,False,True,False,False,False,False,True,False,False,False
2,False,True,False,False,False,False,False,True,False,False,...,False,True,False,False,False,False,True,False,False,False


## 2.2 Khai phá Frequent Itemsets & Luật kết hợp

Tham số từ `params.yaml`: min_support, min_confidence, min_lift.

In [4]:
miner = AssociationMiner(cfg)
print(f'Tham số: support={miner.min_support}, confidence={miner.min_confidence}, lift={miner.min_lift}')

freq_items = miner.mine(basket, algorithm='fpgrowth')
print(f'\nTop frequent itemsets:')
print(freq_items.sort_values('support', ascending=False).head(10))

Tham số: support=0.05, confidence=0.5, lift=1.2
[association] Mining with fpgrowth, support=0.05


[association] Found 298 frequent itemsets

Top frequent itemsets:
     support                                   itemsets
0   0.800525                       (WeatherType=Cloudy)
1   0.509228                          (Vis_bin=HighVis)
6   0.502659                            (Wind_bin=Calm)
2   0.462060                      (Press_bin=HighPress)
3   0.459327                      (Humid_bin=VeryHumid)
21  0.439947      (Vis_bin=HighVis, WeatherType=Cloudy)
4   0.387409                          (Wind_bin=Breezy)
96  0.376458        (Wind_bin=Calm, WeatherType=Cloudy)
22  0.368534  (WeatherType=Cloudy, Press_bin=HighPress)
7   0.357667                           (Vis_bin=ModVis)


In [5]:
rules = miner.get_rules(freq_items)
top_rules = miner.top_rules(rules, n=20)
print(f'Top-20 luật theo Lift:')
print(top_rules.to_string())
save_table(top_rules, 'association_top_rules', '../outputs/tables')

[association] Generated 141 rules after filtering
Top-20 luật theo Lift:
                                                antecedents                                           consequents   support  confidence      lift
0      (Wind_bin=Calm, Humid_bin=VeryHumid, Vis_bin=LowVis)                                   (WeatherType=Foggy)  0.053957    0.609666  8.187685
1                                       (WeatherType=Foggy)  (Wind_bin=Calm, Humid_bin=VeryHumid, Vis_bin=LowVis)  0.053957    0.724629  8.187685
2                           (Wind_bin=Calm, Vis_bin=LowVis)              (WeatherType=Foggy, Humid_bin=VeryHumid)  0.053957    0.591952  8.029114
3                  (WeatherType=Foggy, Humid_bin=VeryHumid)                       (Wind_bin=Calm, Vis_bin=LowVis)  0.053957    0.731860  8.029114
4                                       (WeatherType=Foggy)                       (Wind_bin=Calm, Vis_bin=LowVis)  0.054461    0.731404  8.024112
5                           (Wind_bin=Calm, Vis_bin

'../outputs/tables\\association_top_rules.csv'

## 2.3 Biểu đồ luật kết hợp

In [6]:
plots.plot_top_rules(rules, n=15)

[plots] Saved: ../outputs/figures\association_top_rules_lift.png


In [7]:
plots.plot_support_confidence_scatter(rules)

[plots] Saved: ../outputs/figures\association_support_conf_lift_scatter.png


## 2.4 Khai phá theo mùa

So sánh pattern thời tiết giữa 4 mùa – phát hiện điều kiện đặc trưng mùa.

In [8]:
print('=== KHAI PHÁ THEO MÙA ===')
season_rules = miner.mine_by_season(df, builder.build_basket_for_association)
for season, r in season_rules.items():
    if len(r) > 0:
        print(f'\nMùa {season}: Top 3 luật')
        top3 = miner.top_rules(r, n=3)
        print(top3.to_string())
        save_table(top3, f'assoc_rules_{season}', '../outputs/tables')

=== KHAI PHÁ THEO MÙA ===
[association] Mining with fpgrowth, support=0.05


[association] Found 299 frequent itemsets
[association] Generated 66 rules after filtering
  Season=Spring: 66 rules
[association] Mining with fpgrowth, support=0.05
[association] Found 343 frequent itemsets


[association] Generated 223 rules after filtering
  Season=Summer: 223 rules
[association] Mining with fpgrowth, support=0.05
[association] Found 310 frequent itemsets


[association] Generated 202 rules after filtering
  Season=Fall: 202 rules
[association] Mining with fpgrowth, support=0.05


[association] Found 332 frequent itemsets
[association] Generated 447 rules after filtering
  Season=Winter: 447 rules

Mùa Spring: Top 3 luật
                                           antecedents            consequents   support  confidence      lift
0  (Temp_bin=Cool, Vis_bin=ModVis, WeatherType=Cloudy)  (Humid_bin=VeryHumid)  0.058892    0.569590  1.789756
1                      (Temp_bin=Cool, Vis_bin=ModVis)  (Humid_bin=VeryHumid)  0.069581    0.567503  1.783201
2                       (Temp_bin=Cool, Wind_bin=Calm)  (Humid_bin=VeryHumid)  0.084137    0.563039  1.769173
[report] Saved table: ../outputs/tables\assoc_rules_Spring.csv

Mùa Summer: Top 3 luật
                                           antecedents                                consequents   support  confidence      lift
0  (Wind_bin=Calm, Temp_bin=Warm, Press_bin=NormPress)  (Humid_bin=VeryHumid, WeatherType=Cloudy)  0.078170    0.648052  2.548759
1       (Wind_bin=Calm, Temp_bin=Warm, Vis_bin=ModVis)                

## 2.5 Insight tổng hợp

In [9]:
print('=== INSIGHT TỰ ĐỘNG TỪ LUẬT KẾT HỢP ===')

if len(rules) > 0:
    # Top 5 luật mạnh nhất
    top5 = rules.head(5)
    for idx, row in top5.iterrows():
        ante = ', '.join(list(row['antecedents']))
        cons = ', '.join(list(row['consequents']))
        print(f'  {ante} → {cons}')
        print(f'    support={row["support"]:.3f}, confidence={row["confidence"]:.3f}, lift={row["lift"]:.2f}')
    print()

# Insight theo mùa
if season_rules:
    print('Top luật theo mùa:')
    for season, srules in season_rules.items():
        if len(srules) > 0:
            best = srules.iloc[0]
            ante = ', '.join(list(best['antecedents']))
            cons = ', '.join(list(best['consequents']))
            print(f'  {season}: {ante} → {cons} (lift={best["lift"]:.2f})')

# Thống kê tổng
print(f'\nTổng số luật (sau filter): {len(rules)}')
print(f'Lift trung bình: {rules["lift"].mean():.2f}')
print(f'Lift cao nhất: {rules["lift"].max():.2f}')

=== INSIGHT TỰ ĐỘNG TỪ LUẬT KẾT HỢP ===
  Wind_bin=Calm, Humid_bin=VeryHumid, Vis_bin=LowVis → WeatherType=Foggy
    support=0.054, confidence=0.610, lift=8.19
  WeatherType=Foggy → Wind_bin=Calm, Humid_bin=VeryHumid, Vis_bin=LowVis
    support=0.054, confidence=0.725, lift=8.19
  Wind_bin=Calm, Vis_bin=LowVis → WeatherType=Foggy, Humid_bin=VeryHumid
    support=0.054, confidence=0.592, lift=8.03
  WeatherType=Foggy, Humid_bin=VeryHumid → Wind_bin=Calm, Vis_bin=LowVis
    support=0.054, confidence=0.732, lift=8.03
  WeatherType=Foggy → Wind_bin=Calm, Vis_bin=LowVis
    support=0.054, confidence=0.731, lift=8.02

Top luật theo mùa:
  Spring: Temp_bin=Cool, Vis_bin=ModVis, WeatherType=Cloudy → Humid_bin=VeryHumid (lift=1.79)
  Summer: Wind_bin=Calm, Temp_bin=Warm, Press_bin=NormPress → Humid_bin=VeryHumid, WeatherType=Cloudy (lift=2.55)
  Fall: Wind_bin=Calm, Humid_bin=VeryHumid, Vis_bin=LowVis → WeatherType=Foggy (lift=7.33)
  Winter: Temp_bin=Cold, WeatherType=Foggy → Press_bin=VHighPr